# EPU E6 real-data GPU corrected study R2

Frozen before data access. This exploratory successor runs 20 repeats of five-fold stratified group cross-validation on the real Optical Digits dataset, with fixed same-bank controls, paired inference, Holm adjustment, and a shuffled-label control. It refuses CPU fallback and does not modify the retained E5-R1 ledger.


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, tarfile
PAYLOAD_SHA256 = "203d1d233d40a18de5e307139bb891f29db066226b83f42e2f18355261db6bc7"
PAYLOAD_B64 = """H4sIAAAAAAAA/+09a3PbRpL7Wb9iDqm7gAoIk7IlO0qYOsWms651LJ+kZL3FY2EhcihhBQIMAMqifaq6H3G/8H7J9WNmMAOCemyyj6siKrHwmOnp6enp1/QM5WIZyYMn70+Oz45fHr8N/1Lm2e9+46sH18GzZ/QXrubf571eX9/z+35/72Dvd6L3WyPSdi3LKi6g+b9HW/+E1+cdIbxrWZRJnnmHwhu+/6l7Mjx6++ro7Kg7POj+gM97XoDFZkX+SWbRsppE07iSWHyvt3fQ7fe6e30uUi4XAEtOZbkRmFuyxIajQsYlt3/SF/k8qSo5FdWlFHmRXCRZnHZ/WcpiJeJscpkXopLFHO6nYppMRZZXIl4s0pWo8kX3ShSyKhJ5HaffiJM9McmLQk6qUpzn1aU4l7O8kFAVQE1VM4DUE+hOLIplFjJu8jqZymwioyJPqZvDD+/fHp8cnR2f/CnCDkXYo+jl8cnJ8OXZ8FV0+tPLl8PT0+OT6M2718cnP8Kr7/8UQfej/zjqR0fvXkU/96Of9xk6NlbKCuAi8eEFNlYl1QpbKifJVVJ1UxkXmUjzeBpNgQKAf76okkmcikvo98cCUc8EfyKoiPXNAnoqp4D1RyR///nXz5ufZjKulgWNzsGz5sdJGpclfRv1AtEPxF4gngbiWSD2A3EQiOeBeBGIr8eqXppMZFYSfb4/fdV92n2ZxstSCiLmNClhGM6XSOGPCZDe7tg3YiqL5Bo+AUvNxU8v34hj1b0TOckvsqQCrhD5TPze6u0rp7f4VhZRMi2j+DpO0vichmoWp6WEErc8kNC1IpkDeeM0WgJYRFbexJOqqyghLop8uRDxDICJGCZCDPx2AUy0gO+AYL6suvmsO8vTqVgUcppMELWS+pRkXJuHlasg9fZ6PF2gDj7uM79LSU+jfg8p24d/nvaAos/wcb8HdD7Au+f4z4ve10Bn/Nrv4W2/j9/7e/it/5T+fUbv9xFMnyr2n9ObF3RPdfd6vac0Vl6WJ6WMQNBl07gAjgL+jqkbNNQhDHYv7O/jv6rCXFaXOaPrTROcQNF8mVaJFwhPT8kIptsVvuBJaR6vsuwp/k3zi0JeeApgPpVpzfGLIq/yarWQZQQDxIyHhFNja6YwAo2urC+mYjTjwZxKFAZJBuyWTGD0C5AaJYxuUpSVAB6qihg+4ojCpIAX6UozUFnJBTb6Qj3H6eIyhudeeKDenMsKXxyEPfXCdL1GMI3n51Ms1TelFD1aygCVVRkgEvXrqZ5LRKzopQNIvZzHNxHyOnFWr2eYO8lmskAxVdN1kmczFl3U2tf7uid5XsF8jBeAVRnPFynN8j5qW03YGIZ5GpXJRYZDMl9WhkOomC43i+dJuvqI/FQTrKfbISZZgGBQ0uz3eQqCevoXULI4CUU8KfKyBMleSJDCk8myiCcrcbla5Dhs0kzuO6aqkXYw4HNgaCy2zD7K5OIS5+tcxplupqV2+U09tUtAtAqQMQJWJhL4EIjQpeki8vNSFtdMBRGj1iDpABWTrAQiCxlPLpX4UHqlHhEzSFbfYDIR6jxWJPeJRqgZfzp78/bN2Z9U53gkYbxKooI1wQSw+rIU9pys8VLd1v0Q3I9UXsu0rCHjZJkv55GczQAGj1/ffIXKibYFuIT4bsBFQB19BCH59f6/CuYWYdgK7pZAwe8EiBIc8y6POcrPTHaRWiA9a64SC/EtsY1Hzd4GG+lycvz9T6dn70C3/nWkqRQRQLL1/p+S4N3xOzAqhidvjk+6b49/OBn+8FBSzJIbaBSkCAtHkCUF21uPZpm4ALkHjXQfQKjugymlSuLcexjF4gsQ5iDX0TKkyoIxeygpT8+Ovr9jnjmaTdHQoSrKFtCilUTzZpHGE4lS7U6+up+t+kSAhzBW2Fc9hH+VYpcXQJtrGVE3ckvDggUOopxNdOEqSa0Su2AzydQmcQkSTMlH0RNk8/RwDn2SRc6sEQjUaMvYmHWO8teqpCwjFPHlJUCItJgncuzVghEskY/AczXKhfxliV1HpF/+9OoILXPVSAUsCYMkXogfku/BzgdzDkZjnhdGm8/ABAQpH9HwkEn69vjlH8AOf3ccHb368c0ZmukAsEYyAYM3ny/AQo3iJZg7RfKJ2nYtSJgXyTyiIYgL0mnDm0WaF3EFjQOtwDVhexenTL4gLafdh1CcXbI5DOY+GO4l0FCw3SrY5p8l4AMFosyBpROgfkneDBfpAo/9JU8AHmn1Yk5DFMAQTWGEMqyPY1Xk0+WEv5wenx2ZxgPwncQyg1EAMwTNewFMByY26FuDn7dzu/OPdkC31z/0khz/Aaf21Y/DcD79W7RxT/xn7+nB00b8p/es/3Qb//l7XF+IIUjZWo4ND0jsqrAJWsnLyQSsBhAmJ3s7O2copkq5iEEAynQlVOTIKVcs0WAWHC0K0IMGOSu7rtxS7e2AVCuSG/TU/hBfXKQSqoFrwCiQcA7Fm0pMc8miEfzIZLZC0VYiLLIEwNEChSanO8P97glo06qS8wVqjOmFLFgGk9s4yVPxMUb0iyROUdpru52Eb92BHEADuKoW3uHOzsmeqINbIFjljZxQdAObpOjBopBd6LuIl9OkAu25JPxAd7kxrZ1HxbQw9GOss0aAK0TISbmjCSBiLIt6v6vUoaWKAP8EHT/UUCXRjdwj8fM+AAQNTX7Lzp8/fSkGgjy73U/iK+H3u/TQ2fXhlh3Zzu4v8IXvd1n1+586nT8T9kAhIOm83DG+kMiBSVp9sQdFXXbsqMua5xUiS0owypbTlViClwUusgJTYvxoBsYRB2/QliKFO1U+G1m+XSBjMlWaldzSHbaacQIsF2R3sjUMWjoGLj6PsyvlY1B/Ne1IS6MJBgVnyzSlgjtX794FOBUcQ7xrGeLI5DEZE4E2/Bj7LrVae5TBDvnStoOtW2QLOhbl5XI2A65Xdp22DXWxUJBRBZwe76DxRRFQxkJZXsgaoTgSL2H2gRWUnseTKyw/lWlyLtWML+QM6DxVhNdhO47NQRcmV+BXAHNMrsoWUycUw2uZIa482YBSKC6QieWDzargXiMJ5MPOJjMp3Fo89qX0P02gcLH6m7Rxt/7ff37w7KCp/w+eP9vq/7/H5Xnea1LUZAeA9q/nHiph5gtxCjrZWSIMod7ODkXvo2i2RKEeRahg8gLlIkxQdih3dvS74gIEHThW6nlSXuvbi0/JQt+Dk3QJwkY/YlP6Hqb2pb7PS257Ae+guG74vVUE3PMKNZF+LkBI5uYJZGWVpOZpVerbKplLg3S2nC9WqFUzg2B5RSsY3Lx6CJWVUGo8rHUbt2AKajouIoqG68JvlVo4MVrBrTNHdT8xsLU7HZWgokAWnscpWBCy9rP1h1mf7xrQsOmolKkkaaihnhrt+AMqxz+8Rs3rVFwkC4noCzMcVzLSLxtFSd6ifEelblrg1YdTUBeyMOTOi8nlzs7Oz8OT0zfH78D22LQCuXNyfHwG33GMfeC5JAWO64RAsjy9ln4nBPYCWQ+wpnIG4xvv7R9E56tKlj5GLOShoIeO6H6HtsAhhQHAcloWmea6kGtx+U54KW9gDMF/9jsuVGzbR9Y7JGxaQToIYFnAFJiCnzsaICmxKJMfLXCBUPjm53+BQSLo78DEZvAEirsazq/AEPH5oRycFUsMz9wAM0X5FT12qAqFUKgeGra+d+NBsWwClnR2MfCW1az7wusgl+PSYqrawYsm+hTmAFMkUAU4BJFVgz2MXBRVdCVXunmwGvKPURZng9cYRukYWFw1pA773n9mniYBzRVtn/vUWzT3FDHjjzDkPg39E+E5IsjrOCRl6mg7f8DII/DSByj8OZmZEuGFrHyz8N0R/zIQmgfzolFKr+xRKbAv1wrwSh993q/JV8QYS/0ZSTcsirzwvWVmVp01AM9glsLYuGB5yTAQo3GHYOuvI4PR+K7WsDqYRUswgMBpAjuy5OVWFeCjL6p5FkiDRrd40S4Qn28NkvRKU8VdpGPqPLD7a5VbGti82kZN9UMaCavCpoU3Kt4L+w/GTbtDyskpFXpqamsiKQZWxnN0sVg22Rf6g6YqCblwspzGYWKtVvudJkInMCKgfxRKGyKYh2zHA8Jm2dswP/hUA7s5IAut9k4kTjHwzaoE5kqPy3MYdQCzufKpbkhvIg6smhHhYt+KF2JX9Ht7z3Z3n/6VeNtxW3C9QVJwQNfT+Bi8S4N3jSx+nMfZMkb1Jac+pp4APv21ylahCORRa0FwYSInKg4lL4Dhqst56deiU434Z9Nfj7HCODC2dVgvKmE0HhxE+MK0xAfrm01aFlmeIohVCEHSa0RRySbqkwMIPppCUaSKRVGg1gaYK3EFNZqAlJZFlCkBXh6CIROiFi7iFbFq/chDysWAJ+BDXNIHVTUQU1x2H8CHGUjV6uAZU4ibgFkz0JW76ibE5RI/Bn006AfiSsrFNJmXll7KME7AbaU4zy9CfONriKBMNtfFmQUoZiufoXwL8kB2+3trU8qe42h7dcFPxESICtd9xYRMVeBFQoaDQu5kN/17wvja5KUhMKTlATmTGehEIq794vBBxIJOWv1tIRWDVNS6BusAxF2DaBthAMnOc1CxDKOFdI+gXUWd+nUknC4xjhGD/cPRIP/Dvew5jxcLMFoOScSOaBahMVKNgTSfb1Wb5TKt4Hk0pueZQjPJbJ7+0MLOde/BngEAUAvKYynf+3b2AvTgJF+slF0DotK2OxR9sSISAxrTqJrPFv4jKIcoo8ZXr2ow3IEQXsts6ts1HKJaneEaVo+AIjg9mcwzYNQ63cYlciBW7qNJxjlEsgKKe8zJMFRyZBes75UJ8iGD0g2h88Eyyqj1QFBoqqThCewh4pAV0A0NShDuAB+A/bKU/gppnSZohVtDtMCYD8L5kI1gsAYMYGy+01LtQBWzJFGvJjSnCLH6Q2oWF/P4xm8II9VOlwBqidTp1FDA4iglNjUieDUGHy/BS6Ah5iIdUKE1eR22wFw59OGUMASV5DvfSQIsQig0ufJHmzDkmxHa5jdjgypRl94hdRmV+muw1o6Suc77Tju2IwUNsO72nSL8QfOwS2BT3aZizSKhvKmwktObtk7UlZmndMURNkevOmOwWAzJufz7dS61ZpLNqWvywZl+7wN7BjIGm2ZgnbOmQtk8uhT9d6ZfY7asf+N21t+jDXyILig/so2iNRI/1QVTFp/Bzr1Tm2Ur8+ovRvvEZcTi328QkjrU0VTgwop4gcJpwH+UXGgBuT4C98MBqrRAcsdEKc48u2gFkWDGKRnDv4QTKCV9xyFBAePmQdYTmPNBoKJ5g3a2SqcA1UqvARPLr9ZQlar4fHdK5a0rLNZU8wx8l6ss/5hZaxHMEJ/xz61n6xbXM0LfmlhidIezNXao4KK6Jj7sCzstFPhNrtm4Ro7WmQAjGmqNFucYWqUwJ7NZiPI0rTLkUirJroo0/U2r9PoAorSJKBUmzi6kb4HhnNGxrYkIBMbZcLQZ1r+LRXjmFPhCHKlFIJZiIBfjtFpREE3nKRIQgQ6am68TOpB0TcPwjOMCY1cLOeqNNcczTiE9NHke5KKaVkk2y327aCeEHjotAm64iqQbpOU+1eGuRgfszQER1tifKYh7WVYqLMRanY1REI4gf2RbI5EhJLd1EVeXslCtUUY6l3MVEud+1rXKfFahpiFW2XVBK/xcAIo5yC5XIJZzX4EFO6QEySY/SX8PCbcYMcBxKyjNTJZ/sK5iidF3uezaV1xzDXtAWl53hXK+etGcPPhNC66v1uf2bt0vF8dOu6RqYXM2z2i6b2TxBzOkDedBbLleoWbOelFYkO030o2TjeGgbDEpDxjbaHYG312eFfe3qyntjLoJCGlkwgkGgUJaN/BRFVqNNL/VxFd2AibOGxNh3TrgvD33VYtB0Kr9W72pv4FKv7kfJPXiESBXv1a3G879Bbj25m6uvfntxSiIxftl59PHys3rvLLkZJaFs2VGHBinIZgv0WVe+SuYFAvoCHCc3toz6Pc6JNzWmdiZQARes7rLuIpZMRt01fQmKUX0kJUzIC/llPzIVu5LZirZGKyKnhV5WHNv23z1EF1xZaMV2QX7TbzCBxw/i8FGi+C9jyg03eZJmjhGfxt8lKcZO1hx6tMOGZX++oE5pIObZXCTVNjTFKn3WEwSExRCx3VEUGEgzN4LokqTQGyJK9pxFSYLofmYmNzDKDK5BLuxrCMl53E1uUSXf79n3HJcG69qS6hn9SHgCpYlVCafUPWh+0qfrMLA7QTKipYwM2OcBQgNNJAXsiixBXSZOQQUEMiBj//a7x33GzuhPU36PFKgx3YAUlWZFvFH5WVP8gyDTxn87zMUh1HYwIRyvyzjrMIFP6pLw7633+kEd5X4+vm+WeKz0qujRTtf2BnYTdbA6Zumak4J2oZA3ELPj+cPGAiER/V4gwtZPdwXqstU6zyKk2iNCXBz+KjPe5U2MZLd6YfwkpOmvsZOmPSmmQk8DVw4sMgx6vbVZB0/iKmoP1+Z0BAKTJ9b2GVqdxzewjx+TUw3TOArUAIKPRG+0wd8qXjkMk/nEe9+8BeaRSjGiT69EgZmbcl+yRTLi6kKK6vwmQISJpWcl2hsXMnVQNmF+O6Q/h31TUyRI4jFMqO9chhNJU7noIEKVapWOnVcNc7A4PcBtlqz7uAIS1AUlCxmKljS3TSAWkY9BTTMNEL+HO1cAIu6lUE2I6M6eKoq29TmAoqmnDaBe2F9XvBU0wnXavXMctQV73nkMArYWktcj28GWqyrNvoa5pdl8tlfxH/RKj5ObRA901K121zwXltwYuS9Q7Vsa60EYU9wvwL8sRehsFPwmvVV/Z67Bx/4xq7hET18HGPqd8eKC3rW9g0lJZw8E64RWORwaq9lpxgwG/JW7oE3xw1L0axvwOgsl/V6gd7lNOBangsI5m/k7OI5pPFBu8RxDZKS31M0w8h8tVxTl3OAyzRelLibkocaQKs7d43uXGaTy3lcXNHa8Ua7veaO2Fh8KI8z39/rPXuBG32fvei02r7nj61gr8GuALsiz0BOKkuLZC57yclchpjqG5GAAx9d2xRycgXiEoog0YyksKIpLyxxYBWPwTg/vx8FRdl2FLRakFOWirZ3i6vXe/g/dH1396kz2VrGS70J9D5T3B5fwejTjmEL8BOD0RPRl/09V0x4uoOGXfUL3GDfG5N8Bl2rGaKU6Qx8HFzrcAfeymtp5MuwN9CybDsa9UJUeTDa8P/TENdbnqonfNujNz31hv/im/qJyoxNjJXXbewWsA8UldF6xI7eLkyVxgqUtk34u5uYw0sAZGFjzoYPLiHYLJhvo4L8eikIv9rtN5cuj8DbKXCI1PKlgc7Z0Jj4jinyZqlSJaQP1lclocPXauVFY85/wUkcdwz6XBreIWbqodsfI+4oV0tZ+fy2Xs2iTjy7D3ODEENF7elgbjZNDhwz4rMX632tIIX5FrxMb8Lb8OqMHgxC+xrIyMN477cDUb84b76YeOO1leIm0s193w7GHEgPRAT/6ZwjZ3lE07mFnQLRPI/AFpe+B66q17Kq1XrV6VSc6KSYmEPVvJRey/fAFScb1nTqybgJe/ewhDbk1/Ei0GbAuP/1JNH4qjfAcE7UyZRz6vEuF90fJ7dMNWqigPeNdb1VyEqcap9lRuBiUG1JR8S8Pzo9jU6Hb19HZ8PTM1z70JhEnEKJytNKu9yQCtiQu/WBNnW/9LuxlrZgQfr5slosKyuP88Eyl2venYNp5UA2dCf+owDpjbBo09qpZNZSntHihlM498gFgAshyqDw0EJuMy9URqhJO+VerBO05sLNNcyo6SqW+gLPBqt9iMAlDiiGZ+VE+8wH0Ydo5eanUHFL/NMJNuKAVcCdmQFYnG7ZyOj37s5m0Xty9ck7gjJbMDJZlS7LfnB1nurQul8N6v9A+UsNX3y1vi5MIwsoRJxfjH6bnS7MZLBzT6x8E/AbV/W3xPl2ny4jwHXzSAFw9Tz7vCBwK5wjgmgHtDoVqEHgNTHrTeIMjLVJnEbU9XoGW51FCd7Ay/I/6tHlT522ZlSeOUoGlXFupcHdbuRXTCg/HRpuVRTYzN/Ddz+/OTl+9+Pwnalj+WWLFbhRJGHURoOQ32hM0Ov29DenmLrxbY+Bgqse+Yt2St89fQ3uTQck4NZWe30bcLJhdJ2UyXkqVbIlGbdlKLPrBKxuTqrFZNPo5zenb75/O4xeDX9+83J4quXtrZrmaouCCmuhCB3XcUUOAUZKTSDDGcx8FVRll7RD3cfNc77fkurMwZoVht7Qq8Z0nPbwJ17odHB4OcksHbD5ICTOMWFE3Dr6LKSx8t2sPjXWkP45O+ZUrTvoP+5gp47Ve1AnUe1nRxSgI/eD5R/ImAi3G8CLejKdHP2RJlEaXnzitAHaloB7f3hbgq4F7d5ULTsUgDjyI+41GdDuAdyvAFUOnd6pEAlFT91QVE0pTqofN1bwJ9eAbdsmGD+L6JygclCD4HR/GDe141KpfY6R8gkUgzo6aiNIsRrBa2ARpQKADQhCke8b4bPJdUhN4woFKBEtEA+b8hA1Z+1fjCzg4474N+eTaarZf33d54O4p6aBQLqqD5GJqXteZw3wh4hwgl7wDabtOWjCF+e5DQKpjhX95fp1X6h6/bhW27HEjeG+yT9tYrveHz6Xi4KY1h6o9YV+vNxtT/4G32R9D5j/ckARUX341wCP/sIFwRQk/MBLz2cXuDVknefWG9jUgXCGvHVvbx8vcdpZS1WQZASjvm/supp5n9UOnNsnn3kmww2yFPyhyrdeSFIBLGdnd9bosH8wxnPw1pHHi7fZD/SSpWIlJaFrrNorW7vP7ZC8lf9a5x43L1s9bKitosJ3AAFPAaRgy3rA5iq/TtC3jx6jMrm6I/h3N/ma7nS763+n504DGdw5nbXK1cHNTd5027W5F3eGJO/udjlihMbUXf12Y0WHZayq9vs7xodYxarWHiTFcdwIZINy/0qnMq43+iimsCnDR0UinuupL8Ga1tBD2g73UUOkCVUj8EhCucNUgzFB738kVfXBm+R0k6B3KHsPSazav44oFqCNZLnX5t4sjJwMsJb5trGiclbMDovmOmGgrDTHhNemh7PA9LAw4x00UqgGzZnb2SxZXG9q1HA0xiPdB8sewixfm1oD1ZeODiO0+RnNq7EsVgsZdWZZvUP5HsxtKA9Fv7XljQyV5hPM07tI83PQcWbbQG1Y18DbtpY0L6BzXkwd967tWluodRdoBUZZ4MHG6m4GaqzkCo80nIqV8AiOqKvjewLenjU78LQ9vdDN+z7XJo4Gev/kvL2n3cbyqmmYl8JaB1U3/rip9esMnju60W5f4YUROt7MbnbJY1wTWeX+XfHiK6G2wGvs6xwKOoPRneU6jcPZQYCTDqOMaYrzNylnmKAq1dpJk59b3bqZR+f/ZBfOIdeqzQzcEMLsM6Cm9xBsxtRZcvh7ouuEYGx8CSDHEiMTEV0LL+6oKLtawIv0caB+KmeVygUpMA9c3fPBGXIa0aRsJnK5uR+42xLaVEPe/c4ua2U65cuCIvxtMaVmMgIlIbjMUUe5Gzsf9YCxqw5D5RCjQfBcp8XN0rjK8gxP/PQ12dSCpjspkUCRvfBsJ9SNuFdGvmNhkO6HdNpyOcZTOzDI73fVBjXjXzbo2xAENBIPbpRK/xatOjsyWXhZvVcJc5i4ZKGn0+iczTd2khK+4ubE4BGBPD451nEHLbYaO2rKPVr6sMnizYPbG1KTkbOD1M0TmR8LcYTny68BbDnf+F7AyrbciGR90O86qHvOr2eI7lzmVSqdvIP/mqQllPJoYDWO8W45vnpzdzt12NQZP4B92MgfdRhT8QJlxNmObn0M+Xi0+WD3Mf7YQK8HuihxFSAmiA74eGW1pxk3fm1CnmVSTx9aj5cbrU4o+9C1wBzy2aHlDdkMRBuz7orHSVqJy2ZzaGBOkk/KTeFf6zz28dqW77r6CI+NruWoOq950EgAswfAmul0lnMgliAwClyHtRPDHzRmbcf1j/EXIXi4KFZsK1dF38/3MmFjLyGe1Ey0rs+jtknQOM56bB1U/R0eDI2nDuq0EMrK5GO4a6qlD+CfDRj5G1Dik8hdVNoKbDDcNuHcsZAu5W9HpbsKmkY0S2sV4zoafIA5mV2eOVicbwL7lx2+3sdfGLB5b9zIiqBZZIsTdFbwHa30QQu4ZEnTDr9oMtUfXWhuDodHOVxMKZII3uujN2+RB1n+Kr43qKsZYEDqpcQvxGvaJ3nPYZYiNweVo6nCPru6ZbWO97x1UoWc15aARj0egges/ew/cK1nw8IO2irypvIxkL9pWUen0lP3TFKsK+Xs0/BNQuVez86bV2SDeu6Sir1jBq/2nP79HosYq6FOqGD6Gra1bR8zm1pWUBqLOy0VKb9qYwJYY3WHmllT1BsDvHYiv0tN14ZrJB6vGo1i5rol1O+L4DWP4nctMdVWQkvsTbSaaclrudJa3zQrOknH5sR9J+tp7RcCyCRoP6B/bFs4zYojPeHRxqyn/HoxtwuUPLheqEZ2bIuLnXpl2ErEeH389lX04/Ds5M3L03BSXnsd6yQ7syr80FPtYI6nUzIfdMhGh2oCHXMJTLo8viIbUfcnaMtpbwsg1KnqQVu2ebCeJW6dHcIH5w7wlMzwFXDcH+mFr8/foy7gAVPlgHvTaVTl0MSljKcYtP3GeYu+kIp6qnrlcg5qc+VybJ1Cp1bl7Pz2xu+j1ezmfrHku/WrZ+rO+qYVIKafqFs7N2b9Ry6ar6zSxpDSv/e1wcair+PWitpjWFLYSqcmOe6zM/PaRQO6Hq0f7twoYGUH1uns9hFhzd+jqLvX+OTM5rb8pt//6f3x2e+Hp8NTk8DHjKBSvmFW0TSpG/+i/ZBYdmlxOdrzgrsQwu/1PPReqRw0Xsc9FH/+bOWJ3f45dApjQ1YUW9HyUHxuJ/JtqNDxvvhCvC9kt1zICR//bbitxsdyJCo5R92qC9Vyg+hhNIjXFbu7n2kX1ZfJ9Mvxrfjf//4foV6wpISXu7va3tGf+OnL8WF4MLv9Rmw4QWPm4S/BOGemd9UvTZmfhcE9k8V1nG6GMVJt2jbil2j2UOOBaP/c58/j0LP54CuUl4qc7xrGmOcOrG7+1DXh1C9zKbvm85pO+NJRHIpCh20FDXl5jHnwLMY+GZ7+9PbsNJxPQVMw51dofmHMNcTTwn3qU6dFYRCoeZwlM7QFMOO4rNfZyQTDxKb6nCqVBVxgWN/3dr2OG/WkU1eTklOXO3xAPr5C2Y0pq96PR+/evDYptM3Twmw0MNajD5JNmSBVrtrvdMZ1Dqk5ndbtzH3S3TboXx7/+P7t8GxImjHhH4tzsbEkEkfdML7v6VPEGznb7pG96GS4ycYPyvC2ovQex9lLR/qZdDqxq/Lp1M16iOivl9f/AOXyGwp8l9kCM6ROcr7SAXoXJdjx/toRxEVJ1ok+0zs8Ki6WaNW8py/6LF56COMpGErqu+91u5gk3kVL2+PTXPJsAHwDBnhUgWfl3VmXu4L7JjA98r3hcShDUVSuRX+wXlmf4oRPoclPt86RK3AErCUcK4e9uY6zFst1gCsyq+2CVguMlORMOt0HLKd/70ossxRXWyza4Gf8lQV9Mmqxsk9IUHF+3L9gNd3Z3Cm9v/fh5zar3XY3E7moxJD+0O/G4WFFE0fE2Z2n6Er9HNJGCGcliMSo+m0STD2/Ww6hTzB8RbYyki/CYUcbCRetAY9OGEUoSSkRGnR1GV/gZxSU+PW2rVWdmWqj/YRb+ulkyDPDqafWruzqG/rlTju7QqCrWxyES1wwyQC87gSFyaIIp1wUefrES5x/25+s2F7ba3ttr+21vbbX9tpe22t7ba/ttb221/baXttre22v7bW9ttf22l7ba3ttr+21vbbX9tpe22t7tVz/B4uXFCIAoAAA"""
payload = base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest() == PAYLOAD_SHA256
root = Path("/kaggle/working/epu-e6-source")
root.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
    archive.extractall(root, filter="data")
print(json.dumps({"status": "PASS_EXACT_SOURCE", "payload_sha256": PAYLOAD_SHA256, "bytes": len(payload)}, indent=2))


In [ ]:
import json, os, subprocess, sys
source = root / "epu_e6"
output = Path("/kaggle/working/epu-e6-realdata-gpu-r2")
print("Free-compute policy: paid compute is not authorized.")
subprocess.run([sys.executable, str(source / "study.py"), "--self-test"], cwd=source, check=True)
subprocess.run([sys.executable, str(source / "study.py"), "--output", str(output)], cwd=source, check=True)


In [ ]:
import json, shutil
output = Path("/kaggle/working/epu-e6-realdata-gpu-r2")
manifest = json.loads((output / "MANIFEST.json").read_text())
hypotheses = json.loads((output / "HYPOTHESES.json").read_text())
archive = shutil.make_archive("/kaggle/working/epu-e6-realdata-gpu-r2-evidence", "gztar", output)
print(json.dumps({
    "status": manifest["status"],
    "version": manifest["version"],
    "records": manifest["records"],
    "gpu_prediction_batches": manifest["gpu_prediction_batches"],
    "outcomes": hypotheses["outcomes"],
    "negative_control": hypotheses["negative_control"],
    "claim_boundary": manifest["claim_boundary"],
    "downloadable_archive": archive,
}, indent=2, sort_keys=True))
